# Computing climate indicators

This tutorial introduces the main workflows for computing climate indicators with **earthkit-climate**.
We start with an indicator that requires only a single input variable, then move to an indicator with multiple inputs, and finally to an indicator that requires a threshold calculated from a reference period.

In the **earthkit** ecosystem, data retrieved via [**earthkit-data**](https://earthkit-data.readthedocs.io/) (as `Field` or `FieldList` objects) can be passed directly to **earthkit-climate** indicators. Format conversion and unit checking are handled seamlessly behind the scenes without requiring manual conversions.

We will work through three examples:

- **Single-input indicator (number of hot days):**
  Compute an indicator directly from one temperature variable and a fixed threshold.
- **Multi-input indicator (daily temperature range):**
  Compute an indicator that combines daily maximum and minimum temperature.
- **Indicator with a derived reference threshold (warm days):**
  Calculate a percentile threshold from a reference period and pass it to the indicator together with the temperature data.

Along the way, we will see how data loaded with [**earthkit-data**](https://earthkit-data.readthedocs.io/) can be passed to **earthkit-climate**, how indicators with several input quantities are handled, and how derived quantities such as percentile thresholds can be prepared for an indicator calculation.

In [1]:
import earthkit.data as ekd

import earthkit.climate as ekc

## Single-input indicator

We begin by loading daily maximum temperature data (`tasmax`) in Kelvin using `earthkit.data.from_source()` with the `earthkit-climate-sample` dataset source and converting to an xarray representation of the dataset.

For a comprehensive guide on reading data from various sources and formats, see the [earthkit-data documentation](https://earthkit-data.readthedocs.io/en/latest/tutorials/index.html).

In [2]:
tasmax = ekd.from_source("earthkit-climate-sample", "synthetic-daily-temperature").to_xarray()
tasmax

<xarray.DataArray 'tasmax' (time: 12053)> Size: 96kB
array([286.172175, 282.396019, 288.025119, ..., 280.591046, 285.000216,
       292.76872 ], shape=(12053,))
Coordinates:
  * time     (time) datetime64[us] 96kB 1991-01-01 1991-01-02 ... 2023-12-31
Attributes:
    units:          K
    standard_name:  air_temperature
    cell_methods:   time: maximum

We compute the annual **number of hot days** (`tx_days_above`) where maximum temperature exceeds 30°C.


In [3]:
ekc.indicators.tx_days_above(tasmax, thresh="30 degC", freq="YS")

<xarray.DataArray 'tx_days_above' (time: 33)> Size: 264B
array([ 2.,  8.,  9., ..., 14.,  3.,  6.], shape=(33,))
Coordinates:
  * time     (time) datetime64[us] 264B 1991-01-01 1992-01-01 ... 2023-01-01
Attributes:
    units:          days
    standard_name:  number_of_days_with_air_temperature_above_threshold
    cell_methods:   time: maximum time: sum over days
    history:        [2026-10-07 22:37:13] tx_days_above: TX_DAYS_ABOVE(tasmax...
    long_name:      The number of days with maximum temperature above 30 degc
    description:    Annual number of days where daily maximum temperature exc...

> **Automatic unit matching**: the input temperature dataset is in Kelvin (`K`), while the threshold parameter is specified in degrees Celsius (`"30 degC"`). Earthkit-climate automatically converts the units behind the scenes so they match.

## Multi-input indicator

Some climate indicators require multiple input variables.
Here we compute the annual mean **daily temperature range** (`daily_temperature_range`), which combines daily maximum (`tasmax`) and daily minimum (`tasmin`) temperature data.


In [4]:
# Create a fake tasmin array
tasmin = (tasmax - 10.0).assign_attrs(units="K", standard_name="air_temperature", cell_methods="time: minimum")

ekc.indicators.daily_temperature_range(tasmin=tasmin, tasmax=tasmax, freq="YS")

<xarray.DataArray 'dtr' (time: 33)> Size: 264B
array([10., 10., 10., ..., 10., 10., 10.], shape=(33,))
Coordinates:
  * time     (time) datetime64[us] 264B 1991-01-01 1992-01-01 ... 2023-01-01
Attributes:
    units:           K
    standard_name:   air_temperature
    units_metadata:  temperature: difference
    cell_methods:    tasmin: time: minimum tasmax: time: maximum time range w...
    history:         [2026-10-07 22:37:13] dtr: DTR(tasmin=tasmax, tasmax=tas...
    long_name:       Mean diurnal temperature range
    description:     Annual mean diurnal temperature range.

## Indicator with a derived reference threshold

Percentile-based indices (such as `tx90p`) require a calendar day threshold calculated over a multi-decade baseline reference period (e.g. 1991–2020).
Here we compute the rolling 90th percentile threshold and pass it directly to **earthkit-climate**.

### Calculate rolling 90th percentile threshold

Compute the 90th percentile threshold across a rolling 5-day window for each day of the year over the 1991–2020 reference period:


In [5]:
tasmax_ref = tasmax.sel(time=slice("1991", "2020"))
per90 = ekc.utils.climatology.rolling_percentiles(tasmax_ref, p=90, window_width=5)
per90

<xarray.DataArray 'tasmax' (dayofyear: 366, percentile: 1)> Size: 3kB
array([[288.85755162],
       [289.19069297],
       [289.51403079],
       ...,
       [288.28108614],
       [288.63946904],
       [289.13474028]], shape=(366, 1))
Coordinates:
  * dayofyear   (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * percentile  (percentile) int64 8B 90
Attributes:
    units:               K
    standard_name:       air_temperature
    cell_methods:        time: maximum
    climatology_bounds:  ['1991-01-01', '2020-12-31']
    window:              5
    alpha:               0.3333333333333333
    beta:                0.3333333333333333
    history:             [2026-10-07 22:37:13] per: percentile_doy(arr=tasmax...

### Compute annual warm days indicator (tx90p)

Evaluate the `tx90p` indicator using the rolling 90th percentile baseline:

In [6]:
ekc.indicators.tx90p(tasmax, tasmax_per=per90, freq="YS")

<xarray.DataArray 'tx90p' (time: 33, percentile: 1)> Size: 264B
array([[35.],
       [39.],
       [35.],
       ...,
       [48.],
       [29.],
       [35.]], shape=(33, 1))
Coordinates:
  * time        (time) datetime64[us] 264B 1991-01-01 1992-01-01 ... 2023-01-01
  * percentile  (percentile) int64 8B 90
Attributes:
    units:               days
    standard_name:       days_with_air_temperature_above_threshold
    cell_methods:        tasmax: time: maximum tasmax_per: time: maximum time...
    climatology_bounds:  ['1991-01-01', '2020-12-31']
    window:              5
    alpha:               0.3333333333333333
    beta:                0.3333333333333333
    history:             [2026-10-07 22:37:13] tx90p: TX90P(tasmax=tasmax, ta...
    long_name:           Number of days with maximum temperature above the 90...
    description:         Annual number of days with maximum temperature above...